## Simple Calculator Evaluation
Here we can use the Python SDK to develop the simple calculator agent, then save the agent to a config.yaml and run it from there.

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [ ]:
from nat.llm.nim_llm import NIMModelConfig
from nat.llm.openai_llm import OpenAIModelConfig
from nat.tool.datetime_tools import CurrentTimeToolConfig
from nat.tool.datetime_tools import current_datetime
from nat.utils.sdk.nat_agent import NatReactAgent
from nat.utils.sdk.nat_llm import NatLLM
from nat.utils.sdk.nat_tool import NatTool
from nat.utils.sdk.nat_tool_group import NatToolGroup
from nat_simple_calculator.register import CalculatorToolConfig
from nat_simple_calculator.register import calculator

llm = NatLLM(
    config=NIMModelConfig(
        model_name="nvdev/meta/llama-3.1-70b-instruct",
        temperature=0.0,
        max_tokens=1024
    ),
    name="nim_llm",
)

current_time_tool = NatTool(
    config=CurrentTimeToolConfig(),
    function=current_datetime,
    name="current_datetime",
)
calculator_tool_group = NatToolGroup(
    config=CalculatorToolConfig(),
    tool_group=calculator,
    name="calculator",
)

agent = NatReactAgent(
    tools=[current_time_tool],
    tool_groups=[calculator_tool_group],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

/Users/spastoriza/Documents/Programming/public/nat-official/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
agent.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


functions:
  current_datetime:
    _type: current_datetime
function_groups:
  calculator:
    _type: calculator
llms:
  nim_llm:
    model: nvdev/meta/llama-3.1-70b-instruct
    max_tokens: 1024
    temperature: 0.0
    _type: nim
workflow:
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_datetime
  - calculator
  parse_agent_response_max_retries: 3
  _type: react_agent



In [4]:
await agent.prompt('What is 4 * 50 plus the current hour?')

'217.0'

In [ ]:
judge_prompt = """
You are an intelligent evaluator that scores the generated answer based on the description of the expected answer.
The score is a measure of how well the generated answer matches the description of the expected answer based on the question.
Take into account the question, the relevance of the answer to the question and the quality compared to the description of the expected answer.

Rules:
- The score must be a float of any value between 0.0 and 1.0 on a sliding scale.
- The reasoning string must be concise and to the point. It should be 1 sentence and 2 only if extra description is needed. It must explain why the score was given and what is different between the generated answer and the expected answer.
- The tags <image> and <chart> are real images and charts.
"""  # noqa: E501


In [ ]:
from pathlib import Path

from nat.data_models.dataset_handler import EvalDatasetJsonConfig
from nat.eval.tunable_rag_evaluator.register import TunableRagEvaluatorConfig
from nat.eval.tunable_rag_evaluator.register import register_tunable_rag_evaluator
from nat.utils.sdk.nat_evaluator import NatEvaluator
from nat.utils.sdk.nat_general_evaluator import NatGeneralEvaluator
from nat.utils.sdk.nat_targeted_evaluator import NatTargetedEvaluator

path_to_dataset = Path(os.path.curdir,
                       "../../../../",
                       "examples/getting_started/simple_calculator/data/simple_calculator.json").resolve()

evaluator = NatGeneralEvaluator(
    max_concurrency=1,
    output_dir=Path(".tmp/nat/examples/getting_started/simple_calculator"),
    dataset=EvalDatasetJsonConfig(file_path=path_to_dataset),
)
evaluator_llm = NatLLM(
    config=NIMModelConfig(model_name="mistralai/mixtral-8x22b-instruct-v0.1", temperature=0.0, max_tokens=1024),
    name="eval_llm",
)
openai_llm = NatLLM(
    config=OpenAIModelConfig(model_name="gpt-3.5-turbo", ),
    name="openai_llm",
)
tunable_rag_evaluator = NatTargetedEvaluator(config=TunableRagEvaluatorConfig(llm_name=evaluator_llm.llm_name,
                                                                              default_scoring=True,
                                                                              default_score_weights={
                                                                                  "coverage": 0.5,
                                                                                  "consistency": 0.3,
                                                                                  "relevance": 0.2
                                                                              },
                                                                              judge_llm_prompt=judge_prompt),
                                             name="tuneable_eval",
                                             evaluator=register_tunable_rag_evaluator)

evaluator = NatEvaluator(general_evaluator=evaluator,
                         evaluators=[tunable_rag_evaluator],
                         evaluation_llms=[openai_llm, evaluator_llm])

agent.add_evaluator(evaluator)

In [8]:
path_to_yaml = Path(os.getcwd(), "config", "eval_config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
agent.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  current_datetime:
    _type: current_datetime
function_groups:
  calculator:
    _type: calculator
llms:
  nim_llm:
    model: nvdev/meta/llama-3.1-70b-instruct
    max_tokens: 1024
    temperature: 0.0
    _type: nim
  openai_llm:
    model: gpt-3.5-turbo
    _type: openai
  eval_llm:
    model: mistralai/mixtral-8x22b-instruct-v0.1
    max_tokens: 1024
    temperature: 0.0
    _type: nim
workflow:
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_datetime
  - calculator
  parse_agent_response_max_retries: 3
  _type: react_agent
eval:
  general:
    max_concurrency: 1
    output_dir: .tmp/nat/examples/getting_started/simple_calculator
    dataset:
      file_path: /Users/spastoriza/Documents/Programming/public/nat-official/examples/getting_started/simple_calculator/src/nat_simple_calculator/data/simple_calculator.json
      _type: json
  evaluators:
    tuneable_eval:
      llm_name: eval_llm
      judge_llm_prompt: |2

        You are an intelligent evaluator

In [9]:
await agent.evaluate()

Evaluating RAG: 100%|██████████| 12/12 [00:14<00:00,  1.21s/it]
